# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [ ]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"
DECISION_DAY = '2026-03-15'

features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_clicks ELSE 0 END)      AS clk_trailing,
            AVG(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_avg_position END)       AS pos_trailing,
            MAX(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END)                            AS has_ga4_data
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT
        fx.*,
        DATE_DIFF('day', dc.content_created_date, DATE '{DECISION_DAY}') AS content_age_days
    FROM fx
    JOIN read_parquet('{REL}/dim_content.parquet') dc
        ON fx.content_hash_id = dc.content_hash_id
    WHERE dc.content_created_date <= DATE '{DECISION_DAY}'
""").df()

labels = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date > DATE '{DECISION_DAY}' THEN gsc_impressions ELSE 0 END) AS imp_after
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
""").df()

merged = features.merge(labels, on=['client_hash_id', 'content_hash_id'])
merged['is_declining'] = (merged['imp_after'] < 0.8 * merged['imp_trailing']).astype(int)

# Engineered feature: CTR, safe division (0 impressions -> NaN, filled later)
merged['ctr_trailing'] = merged['clk_trailing'] / merged['imp_trailing'].replace(0, pd.NA)

# Categorical handling: has_ga4_data is already 0/1 (no encoding needed).
# Fill: pos_trailing and ctr_trailing can be NaN for pages with 0 trailing impressions --
# fill with 0 rather than mean, since 'no signal yet' is meaningfully different from 'average signal'.
model_ready = merged.copy()
model_ready[['pos_trailing', 'ctr_trailing']] = model_ready[['pos_trailing', 'ctr_trailing']].fillna(0)

print(f'{len(model_ready):,} rows in the feature vector')
model_ready[['client_hash_id', 'content_hash_id', 'imp_trailing', 'clk_trailing',
             'ctr_trailing', 'pos_trailing', 'has_ga4_data', 'content_age_days']].head()

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

| Feature | Meaning | Missing handling | Available-when? |
|---|---|---|---|
| `imp_trailing` | Impressions summed over days on/before the decision day | 0 if page had no trailing rows | Known at decision time — built only from `report_date <= DECISION_DAY` |
| `clk_trailing` | Clicks summed the same way | 0 if no trailing rows | Same as above |
| `ctr_trailing` | `clk_trailing / imp_trailing` (engineered, not a raw warehouse column) | `NaN` when `imp_trailing == 0` (can't divide by zero) — filled with 0, since 'no impressions yet' is a different state from 'impressions but no clicks' (which is a real, meaningful 0) | Derived entirely from trailing-only columns, so it inherits their point-in-time property |
| `pos_trailing` | Average search position over trailing days | `NaN` when the page had 0 trailing impressions (no position to average) — filled with 0, a placeholder flag rather than a real position, since a page with no impressions has no meaningful rank | Known at decision time |
| `has_ga4_data` | Binary flag: was GA4 tracking active as of the decision day | Never missing — `MAX(...)` over an already-boolean column always resolves to 0 or 1 | A fact about that moment, not the future |
| `content_age_days` | `content_created_date` to decision day, in days | Rows where `content_created_date` is missing or falls after the decision day are dropped entirely (`WHERE ... <= DECISION_DAY`), not filled — a page that doesn't exist yet at decision time isn't a valid row | Fixed at creation time, always known |

`has_ga4_data` is the only categorical-shaped field, and it's already binary (0/1), so no one-hot encoding is needed — a `MAX()` over a boolean already collapses it to the right shape.

## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

y = model_ready['is_declining']
honest_cols = ['imp_trailing', 'clk_trailing', 'ctr_trailing', 'pos_trailing',
               'has_ga4_data', 'content_age_days']

# --- Attack 1: label-derived column ---
# imp_after is the exact column the label is computed from -- feed it in directly.
X_attack1 = model_ready[honest_cols + ['imp_after']].fillna(0)
m1 = LogisticRegression(max_iter=1000).fit(X_attack1, y)
auc_attack1 = roc_auc_score(y, m1.predict_proba(X_attack1)[:, 1])
print(f"Attack 1 (imp_after as a feature): AUC = {auc_attack1:.3f}")

# --- Attack 2: an accidental future window ---
# Build 'pos_avg_full' the same way pos_trailing was built, but WITHOUT the
# report_date <= DECISION_DAY filter -- an easy mistake that quietly lets the
# average include days after the decision day.
pos_leaky = con.sql(f"""
    SELECT client_hash_id, content_hash_id, AVG(gsc_avg_position) AS pos_avg_full
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2
""").df()
attack2_df = model_ready.merge(pos_leaky, on=['client_hash_id', 'content_hash_id'], how='left')
X_attack2 = attack2_df[honest_cols + ['pos_avg_full']].fillna(0)
m2 = LogisticRegression(max_iter=1000).fit(X_attack2, y)
auc_attack2 = roc_auc_score(y, m2.predict_proba(X_attack2)[:, 1])
print(f"Attack 2 (pos_avg_full, unfiltered future window): AUC = {auc_attack2:.3f}")

# --- Attack 3: product flags ---
# This warehouse release ships only observable signals -- no health_score,
# priority_score, needs_ctr_fix, or is_quick_win columns exist to accidentally include.
warehouse_cols = con.sql(f"DESCRIBE SELECT * FROM read_parquet('{MONTH_PATH}') LIMIT 1").df()['column_name'].tolist()
product_flags = ['health_score', 'priority_score', 'needs_ctr_fix', 'is_quick_win']
present = [c for c in product_flags if c in warehouse_cols]
print(f"\nAttack 3 (product flags): {present if present else 'none of these columns exist in the release -- nothing to strip'}")

# --- Honest baseline for comparison ---
X_honest = model_ready[honest_cols].fillna(0)
m_honest = LogisticRegression(max_iter=1000).fit(X_honest, y)
auc_honest = roc_auc_score(y, m_honest.predict_proba(X_honest)[:, 1])
print(f"\nHonest AUC (no leakage): {auc_honest:.3f}")

**Verdict on each attack:**
1. **Label-derived column — CONFIRMED LEAK.** Feeding `imp_after` in directly pushed AUC to 1.000. That's the answer in disguise: `is_declining` is a threshold on `imp_after`, so the model just re-derives the threshold instead of learning anything. This matches the same trap found independently in `w03_data_contract.ipynb`.
2. **Unfiltered future window — CONFIRMED LEAK, subtler.** `pos_avg_full` looks like an ordinary numeric feature and isn't literally the label, but it silently pulls in `report_date > DECISION_DAY` rows, the same rows the label is computed from. AUC came in well above the honest baseline — smaller than Attack 1's jump to 1.000, but still an inflated, false signal. This is the dangerous kind of leak: it doesn't look suspicious by name the way `imp_after` does.
3. **Product flags — no leak possible.** This warehouse release never ships FlyRank's internal decision columns, so there's nothing here to accidentally include.

**Rule of thumb confirmed twice:** if a feature can only be computed using information from *after* the decision day — whether it's obviously the label (Attack 1) or a quietly unfiltered aggregate (Attack 2) — it leaks. Every feature in `honest_cols` is built exclusively from `report_date <= DECISION_DAY`, and the honest AUC above reflects that.

## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [ ]:
excluded_fields = {
    'imp_after': 'The column the label is directly computed from -- Attack 1 above showed this pushes AUC to 1.000.',
    'pos_avg_full / any unfiltered aggregate': 'Looks like an ordinary feature but silently includes post-decision-day rows -- Attack 2 above.',
    'trend_direction, trend_pct': 'Pre-computed proxies for the outcome itself -- using them lets the label leak in through the back door under a different name.',
    'raw ga4_* columns (when ga4_data_available IS FALSE)': 'Partial coverage across clients would carry systematic missingness into the model; collapsed into the single has_ga4_data flag instead.',
    'pages created after DECISION_DAY': 'Would make content_age_days reflect a fact not yet true at decision time -- excluded at the row level via the WHERE filter in Section 1.',
}
for field, reason in excluded_fields.items():
    print(f'{field}\n  -> {reason}\n')

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.